# Optimizer Quick Reference Cheatsheet

A comprehensive quick reference guide for optimization algorithms in deep learning.

**Table of Contents:**
1. Decision Flowchart: Which Optimizer Should I Use?
2. Hyperparameter Defaults Table
3. Common Configurations by Task (CV, NLP, RL)
4. One-Page Summaries of Each Optimizer

---

## 1. Decision Flowchart: Which Optimizer Should I Use?

```
                    START
                      |
                      v
        +---------------------------+
        |   Is this your first      |
        |   attempt at training?    |
        +---------------------------+
                |           |
               YES          NO
                |           |
                v           v
        +------------+   +------------------------+
        |   Adam     |   | Are you fine-tuning    |
        | lr=1e-3    |   | a pretrained model?    |
        +------------+   +------------------------+
                              |            |
                             YES           NO
                              |            |
                              v            v
                    +------------+   +-----------------+
                    |   AdamW    |   | Is training     |
                    |  lr=1e-5   |   | unstable/slow?  |
                    | wd=0.01    |   +-----------------+
                    +------------+         |        |
                                         YES       NO
                                          |        |
                                          v        v
                    +-----------------+   +------------------+
                    | Try these in    |   | Stick with Adam  |
                    | order:          |   | or try SGD+M     |
                    | 1. Lower LR     |   | for better       |
                    | 2. Grad clip    |   | generalization   |
                    | 3. SGD+Momentum |   +------------------+
                    +-----------------+
```

### Quick Decision Rules:

| Situation | Recommended Optimizer | Key Settings |
|-----------|----------------------|--------------|
| Starting a new project | **Adam** | lr=1e-3 |
| Fine-tuning transformers | **AdamW** | lr=1e-5 to 5e-5, wd=0.01 |
| Training CNNs from scratch | **SGD + Momentum** | lr=0.1, momentum=0.9 |
| Sparse features (NLP embeddings) | **Adagrad** or **Adam** | Default settings |
| Reinforcement learning | **Adam** | lr=3e-4 |
| RNNs/LSTMs | **Adam** or **RMSprop** | lr=1e-3, clip gradients |
| Maximum generalization | **SGD + Momentum + WD** | lr=0.1, momentum=0.9, wd=1e-4 |
| Unstable training | **SGD + Momentum** | Lower lr, add grad clipping |

## 2. Hyperparameter Defaults Table

### Learning Rate Defaults

| Optimizer | Default LR | Typical Range | Notes |
|-----------|-----------|---------------|-------|
| SGD | 0.01 | 0.001 - 1.0 | Scale with batch size |
| SGD + Momentum | 0.1 | 0.01 - 1.0 | Use with LR schedule |
| Adam | 0.001 | 1e-5 - 1e-2 | Often works out of box |
| AdamW | 0.001 | 1e-6 - 1e-3 | Lower for fine-tuning |
| RMSprop | 0.01 | 1e-4 - 0.1 | Good for RNNs |
| Adagrad | 0.01 | 0.001 - 0.1 | LR decays naturally |

### Other Hyperparameters

| Parameter | SGD | Adam/AdamW | RMSprop | Adagrad |
|-----------|-----|------------|---------|----------|
| momentum | 0.9 | - | 0.0 | - |
| beta1 | - | 0.9 | - | - |
| beta2 | - | 0.999 | - | - |
| alpha/rho | - | - | 0.99 | - |
| epsilon | - | 1e-8 | 1e-8 | 1e-10 |
| weight_decay | 0 | 0 / 0.01 | 0 | 0 |

### Batch Size and Learning Rate Scaling

```
Linear Scaling Rule (for SGD):
    new_lr = base_lr * (new_batch_size / base_batch_size)

Square Root Scaling (for Adam):
    new_lr = base_lr * sqrt(new_batch_size / base_batch_size)
```

## 3. Common Configurations by Task

### Computer Vision (CNN Training)

#### ImageNet-style Training
```python
# ResNet on ImageNet
optimizer = SGD(params, lr=0.1, momentum=0.9, weight_decay=1e-4)
scheduler = StepLR(optimizer, step_size=30, gamma=0.1)
# or
scheduler = CosineAnnealingLR(optimizer, T_max=90)
```

#### Transfer Learning / Fine-tuning
```python
# Fine-tuning pretrained model
optimizer = AdamW(params, lr=1e-4, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
```

### Natural Language Processing

#### Transformer Training (BERT, GPT)
```python
optimizer = AdamW(
    params,
    lr=5e-5,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.01
)
# Linear warmup + linear decay
# warmup_steps = 10% of total steps
```

#### LSTM/RNN Training
```python
optimizer = Adam(params, lr=1e-3)
# or
optimizer = RMSprop(params, lr=1e-3, alpha=0.9)
# Always use gradient clipping!
clip_grad_norm_(grads, max_norm=1.0)
```

### Reinforcement Learning

#### Policy Gradient Methods (PPO, A2C)
```python
optimizer = Adam(params, lr=3e-4, eps=1e-5)
```

#### Value Function Approximation (DQN)
```python
optimizer = Adam(params, lr=1e-4)
# or RMSprop
optimizer = RMSprop(params, lr=2.5e-4, alpha=0.95, eps=0.01)
```

### Generative Models

#### GANs
```python
# Generator
g_optimizer = Adam(g_params, lr=1e-4, betas=(0.5, 0.999))
# Discriminator
d_optimizer = Adam(d_params, lr=4e-4, betas=(0.5, 0.999))
```

In [ ]:
# Setup for running examples
import numpy as np
import sys
sys.path.insert(0, '..')

# Import our optimizers
from src.optimizers import (
    SGD, SGDW, Adam, AdamW, NAdam,
    RMSprop, Adagrad, Adadelta,
    StepLR, CosineAnnealingLR,
    clip_grad_norm_
)

print("All optimizers loaded successfully!")

## 4. One-Page Summary of Each Optimizer

---

### SGD (Stochastic Gradient Descent)

**Update Rule:**
```
w = w - lr * gradient
```

**With Momentum:**
```
v = momentum * v + gradient
w = w - lr * v
```

**Pros:**
- Simple and well-understood
- Best generalization in many cases
- Memory efficient

**Cons:**
- Requires careful learning rate tuning
- Slow convergence without momentum
- Same learning rate for all parameters

**When to Use:**
- Training CNNs from scratch
- When you need the best generalization
- Large-scale distributed training

**Typical Settings:**
```python
SGD(params, lr=0.1, momentum=0.9, weight_decay=1e-4)
```

---

### Adam (Adaptive Moment Estimation)

**Update Rule:**
```
m = beta1 * m + (1 - beta1) * g           # First moment
v = beta2 * v + (1 - beta2) * g^2         # Second moment
m_hat = m / (1 - beta1^t)                 # Bias correction
v_hat = v / (1 - beta2^t)
w = w - lr * m_hat / (sqrt(v_hat) + eps)
```

**Pros:**
- Works well out of the box
- Adaptive learning rates per parameter
- Handles sparse gradients well
- Fast convergence

**Cons:**
- May not generalize as well as SGD
- More memory (stores m and v)
- Can have convergence issues (fixed by AMSGrad)

**When to Use:**
- Starting a new project
- NLP tasks
- Reinforcement learning
- When you need fast iteration

**Typical Settings:**
```python
Adam(params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8)
```

---

### AdamW (Adam with Decoupled Weight Decay)

**Key Difference from Adam:**
```
# Adam (L2 regularization in gradient):
g = g + weight_decay * w
w = w - lr * adam_update(g)

# AdamW (decoupled weight decay):
w = w - lr * adam_update(g) - lr * weight_decay * w
```

**Why It Matters:**
- L2 regularization != weight decay in Adam
- AdamW provides true weight decay
- Better generalization, especially for transformers

**When to Use:**
- Fine-tuning pretrained models (BERT, GPT, ViT)
- When regularization is important
- Modern best practice for transformers

**Typical Settings:**
```python
AdamW(params, lr=1e-4, betas=(0.9, 0.999), weight_decay=0.01)
```

---

### RMSprop

**Update Rule:**
```
v = alpha * v + (1 - alpha) * g^2
w = w - lr * g / (sqrt(v) + eps)
```

**Pros:**
- Good for non-stationary objectives
- Works well for RNNs
- Simpler than Adam

**When to Use:**
- RNN/LSTM training
- Reinforcement learning

**Typical Settings:**
```python
RMSprop(params, lr=0.01, alpha=0.99, eps=1e-8)
```

---

### Adagrad

**Update Rule:**
```
G = G + g^2                              # Accumulate squared gradients
w = w - lr * g / (sqrt(G) + eps)
```

**Pros:**
- No learning rate tuning needed
- Great for sparse features
- Per-parameter learning rates

**Cons:**
- Learning rate monotonically decreases
- Can stop learning too early

**When to Use:**
- Sparse data (NLP, recommendations)
- Convex optimization

**Typical Settings:**
```python
Adagrad(params, lr=0.01, eps=1e-10)
```

In [ ]:
# Example: Training loop with different optimizers

def create_optimizer(name, params, **kwargs):
    """Factory function to create optimizers."""
    optimizers = {
        'sgd': SGD,
        'sgd_momentum': lambda p, **kw: SGD(p, momentum=0.9, **kw),
        'adam': Adam,
        'adamw': AdamW,
        'rmsprop': RMSprop,
        'adagrad': Adagrad,
    }
    return optimizers[name](params, **kwargs)

# Example parameters
params = [np.random.randn(100, 50), np.random.randn(50, 10)]

# Create different optimizers
opt_sgd = create_optimizer('sgd', [p.copy() for p in params], lr=0.1)
opt_adam = create_optimizer('adam', [p.copy() for p in params], lr=0.001)
opt_adamw = create_optimizer('adamw', [p.copy() for p in params], lr=0.001, weight_decay=0.01)

print("SGD:", opt_sgd)
print("\nAdam:", opt_adam)
print("\nAdamW:", opt_adamw)

In [ ]:
# Example: Learning rate scheduling

# Create optimizer
params = [np.random.randn(100, 50)]
optimizer = SGD(params, lr=0.1, momentum=0.9)

# Method 1: StepLR
scheduler = StepLR(optimizer, step_size=30, gamma=0.1)
print("StepLR schedule (first 100 epochs):")
lrs = []
for epoch in range(100):
    lrs.append(optimizer.get_lr()[0])
    scheduler.step()
print(f"  Epoch 0: {lrs[0]:.4f}")
print(f"  Epoch 29: {lrs[29]:.4f}")
print(f"  Epoch 30: {lrs[30]:.4f}")
print(f"  Epoch 59: {lrs[59]:.4f}")
print(f"  Epoch 60: {lrs[60]:.4f}")

# Method 2: CosineAnnealingLR
optimizer2 = SGD([p.copy() for p in params], lr=0.1, momentum=0.9)
scheduler2 = CosineAnnealingLR(optimizer2, T_max=100, eta_min=1e-4)
print("\nCosineAnnealingLR schedule:")
print(f"  Epoch 0: {optimizer2.get_lr()[0]:.6f}")
for epoch in range(99):
    scheduler2.step()
    if epoch + 1 in [25, 50, 75, 99]:
        print(f"  Epoch {epoch + 1}: {optimizer2.get_lr()[0]:.6f}")

In [ ]:
# Example: Gradient clipping

# Simulate gradients
grads = [np.random.randn(100, 50) * 10, np.random.randn(50, 10) * 5]

# Check gradient norm before clipping
from src.optimizers import compute_grad_norm, clip_grad_norm_, clip_grad_value_

norm_before = compute_grad_norm(grads)
print(f"Gradient norm before clipping: {norm_before:.4f}")

# Clip by norm
grads_copy = [g.copy() for g in grads]
total_norm = clip_grad_norm_(grads_copy, max_norm=1.0)
norm_after = compute_grad_norm(grads_copy)
print(f"Gradient norm after clip_grad_norm_(max_norm=1.0): {norm_after:.4f}")

# Clip by value
grads_copy2 = [g.copy() for g in grads]
clip_grad_value_(grads_copy2, clip_value=0.5)
print(f"Max gradient value after clip_grad_value_(0.5): {max(np.abs(g).max() for g in grads_copy2):.4f}")

## Summary Comparison Table

| Optimizer | Memory | Convergence | Generalization | Best For |
|-----------|--------|-------------|----------------|----------|
| SGD | Low | Slow | Excellent | CNNs, final training |
| SGD+Momentum | Low | Medium | Excellent | Most CV tasks |
| Adam | Medium | Fast | Good | Quick experiments, NLP |
| AdamW | Medium | Fast | Very Good | Transformers, fine-tuning |
| RMSprop | Medium | Medium | Good | RNNs, RL |
| Adagrad | Medium | Fast initially | Good | Sparse features |
| Adadelta | Medium | Medium | Good | No LR tuning needed |

---

**Key Takeaways:**

1. **Start with Adam** for quick experiments
2. **Use AdamW** for transformers and fine-tuning
3. **Use SGD+Momentum** for best generalization on CNNs
4. **Always use learning rate scheduling** for best results
5. **Use gradient clipping** for RNNs and unstable training

---
*Generated for "Optimization for Machine Learning" book*